# Pipeline - Geração de Clusteres e Hiper-Parametros

Este notebook contém apenas a **modelagem de clustering patrimonial**.

De modo a combater a alta variação de valores a solução final usa uma estratégia **estratificada**. subdividindo os candidatos nos grupos a seguir

1. **Baixo patrimônio:** até **1 milhão de reais** 
2. **Patrimônio intermediário:** de **1 milhão a 10 milhões de reais** 
3. **Alto patrimônio:** de **10 milhões a 100 milhões de reais** 
4. **Ultra-alto patrimônio:** acima de **100 milhões de reais** 

O objetivo é gerar uma coluna final de cluster/perfil por candidato, mantendo a explicabilidade da tipologia.

## 1. Imports e configurações gerais

Altere os parâmetros desta seção caso queira testar novas métricas, pesos, cortes ou números de clusters.


In [40]:
# ============================================================
# CONFIGURAÇÕES GERAIS
# ============================================================
import pandas as pd


CAMINHO_DATASET = "features_patrimoniais_por_candidato.csv"
SEP = ";"

RANDOM_STATE = 42
N_INIT_KMEANS = 50

# ============================================================
# ESTRATIFICAÇÃO PATRIMONIAL
# ============================================================
# Cortes definidos a partir dos quartis aproximados e do P99.
#
# Q1  ~ R$ 78 mil   -> corte : R$ 100 mil
# Q2  ~ R$ 277 mil  -> corte : R$ 300 mil
# Q3  ~ R$ 758 mil  -> corte : R$ 750 mil
# P99 ~ R$ 13,5 mi  -> corte : R$ 13 mi

CORTE_BAIXO = 100_000
CORTE_MEDIO_BAIXO = 300_000
CORTE_MEDIO_ALTO = 750_000
CORTE_ALTO = 13_000_000

ESTRATOS_ORDEM = [
    "01_baixo",
    "02_medio_baixo",
    "03_medio_alto",
    "04_alto",
    "05_ultra_alto",
]

# ============================================================
# NÚMERO DE CLUSTERS FINAL POR ESTRATO
# ============================================================

K_POR_ESTRATO = {
    "01_baixo": 8,
    "02_medio_baixo": 8,
    "03_medio_alto": 7,
    "04_alto": 8,
    "05_ultra_alto": 5,
}

# ============================================================
# NOMES DAS MACROS PARA MELHOR VISIBILIDADE
# ============================================================
NOMES_MACROS = {
    "perc_imoveis": "imobiliário",
    "perc_veiculos": "veicular",
    "perc_ativos_financeiros": "financeiro",
    "perc_participacoes_societarias": "societário",
    "perc_rural_agropecuario": "rural/agropecuário",
    "perc_creditos_direitos": "créditos e direitos",
    "perc_dinheiro_especie": "dinheiro em espécie",
    "perc_outros": "outros ativos",
    "perc_bens_luxo_colecao": "bens de luxo/coleção",
    "perc_direitos_intangiveis": "direitos intangíveis",
    "perc_outros_atividade_profissional": "atividade profissional",
}
# ============================================================
# EXPORTAÇÃO OPCIONAL DE ARQUIVOS PARA AUDITORIA
# ============================================================
GERAR_AUDITORIA_CLUSTERS = False
PASTA_AUDITORIA_CLUSTERS = "auditoria_clusters"

# ============================================================
# BUSCA EXPLORATÓRIA DE MÉTRICAS
# ============================================================

# Se quiser recalcular tabelas de métricas exploratórias, mude para True.
RODAR_BUSCAS_METRICAS = False

# Intervalos de k usados se RODAR_BUSCAS_METRICAS=True.
# O range termina antes do limite superior, então range(4, 15) testa 4 até 14.
RANGES_K = {
    "01_baixo": range(4, 15),
    "02_medio_baixo": range(4, 15),
    "03_medio_alto": range(4, 15),
    "04_alto": range(4, 15),
    "05_ultra_alto": range(4, 15),
}

## 2. Carregar dataset de features por candidato

Este é o dataset usado nos experimentos de clustering. A unidade é o candidato (`SQ_CANDIDATO`).


In [41]:
features_candidatos = pd.read_csv(
    CAMINHO_DATASET,
    sep=SEP
)

print("Shape:", features_candidatos.shape)
features_candidatos.head()

Shape: (18219, 27)


,SQ_CANDIDATO,valor_ativos_financeiros,valor_bens_luxo_colecao,valor_creditos_direitos,valor_dinheiro_especie,valor_direitos_intangiveis,valor_imoveis,valor_outros,valor_outros_atividade_profissional,valor_participacoes_societarias,...,perc_imoveis,perc_outros,perc_outros_atividade_profissional,perc_participacoes_societarias,perc_rural_agropecuario,perc_veiculos,qtd_bens,qtd_macros_presentes,indice_concentracao_macro,patrimonio_total
0,10001595335,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.0,0.0,0.0,0.0,1.000000,2,1,1.000000,60000.0
1,10001595336,0.0,0.0,0.0,0.0,0.0,400000.0,0.0,0.0,0.0,...,0.778362,0.0,0.0,0.0,0.0,0.221638,3,2,0.654970,513900.0
2,10001595338,0.0,0.0,0.0,0.0,0.0,300000.0,0.0,0.0,0.0,...,0.961538,0.0,0.0,0.0,0.0,0.038462,2,2,0.926036,312000.0
3,10001595339,0.0,0.0,0.0,0.0,0.0,250000.0,0.0,0.0,0.0,...,0.714286,0.0,0.0,0.0,0.0,0.285714,2,2,0.591837,350000.0
4,10001595340,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.0,0.0,0.0,0.0,1.000000,2,1,1.000000,32000.0


## 3. Estratificação patrimonial

A clusterização final é realizada separadamente dentro de faixas de patrimônio declarado. Essa etapa evita comparar, no mesmo espaço de clustering, candidatos com escalas patrimoniais muito distintas, reduzindo o efeito da forte assimetria da distribuição monetária.

A estratégia atual utiliza como referência os quartis empíricos do patrimônio total, com cortes arredondados para valores monetários mais interpretáveis. Além disso, foi incluída uma separação adicional próxima ao percentil 99, com o objetivo de isolar candidatos com patrimônio ultra_alto e evitar que esses outliers distorçam a formação dos clusters nos demais estratos.

In [42]:
# TABELA 1: QUARTIS ORIGINAIS E P99
patrimonio_col = "patrimonio_total"

quartis = features_candidatos[patrimonio_col].quantile([0.25, 0.50, 0.75, 0.99])

tabela_quartis_originais = pd.DataFrame({
    "referencia": ["Q1 / 25%", "Q2 / 50%", "Q3 / 75%", "P99 / 99%"],
    "percentil": [0.25, 0.50, 0.75, 0.99],
    "valor_original": [
        quartis.loc[0.25],
        quartis.loc[0.50],
        quartis.loc[0.75],
        quartis.loc[0.99],
    ],
    "cobertura_acumulada_base": ["25%", "50%", "75%", "99%"],
})

tabela_quartis_originais

,referencia,percentil,valor_original,cobertura_acumulada_base
0,Q1 / 25%,0.25,7.800000e+04,25%
1,Q2 / 50%,0.50,2.770000e+05,50%
2,Q3 / 75%,0.75,7.585912e+05,75%
3,P99 / 99%,0.99,1.349856e+07,99%


In [43]:
# TABELA 1B: QUARTIS ORIGINAIS COM COMPLEXIDADE MEDIA ACUMULADA
linhas_quartis = []

for referencia, percentil in [
    ("Q1 / 25%", 0.25),
    ("Q2 / 50%", 0.50),
    ("Q3 / 75%", 0.75),
    ("P99 / 99%", 0.99),
]:
    corte = features_candidatos[patrimonio_col].quantile(percentil)

    grupo = features_candidatos[
        features_candidatos[patrimonio_col] <= corte
    ].copy()

    linhas_quartis.append({
        "referencia": referencia,
        "percentil": percentil,
        "valor_original": corte,
        "qtd_candidatos": len(grupo),
        "perc_base": len(grupo) / len(features_candidatos),
        "qtd_bens_media": grupo["qtd_bens"].mean(),
        "qtd_macros_media": grupo["qtd_macros_presentes"].mean(),
        "concentracao_media": grupo["indice_concentracao_macro"].mean(),
    })

tabela_quartis_originais = pd.DataFrame(linhas_quartis)

tabela_quartis_originais

,referencia,percentil,valor_original,qtd_candidatos,perc_base,qtd_bens_media,qtd_macros_media,concentracao_media
0,Q1 / 25%,0.25,7.800000e+04,4557,0.250123,1.707044,1.285056,0.929408
1,Q2 / 50%,0.50,2.770000e+05,9110,0.500027,2.298134,1.614819,0.853795
2,Q3 / 75%,0.75,7.585912e+05,13664,0.749986,3.096311,1.935304,0.802526
3,P99 / 99%,0.99,1.349856e+07,18036,0.989956,4.805278,2.328343,0.757358


In [44]:
# TABELA 2: ESTRATOS ARREDONDADOS ADOTADOS

CORTE_BAIXO = 100_000
CORTE_MEDIO_BAIXO = 300_000
CORTE_MEDIO_ALTO = 750_000
CORTE_ALTO = 13_000_000

def criar_estrato_patrimonial(valor):
    if valor <= CORTE_BAIXO:
        return "01_baixo"
    elif valor <= CORTE_MEDIO_BAIXO:
        return "02_medio_baixo"
    elif valor <= CORTE_MEDIO_ALTO:
        return "03_medio_alto"
    elif valor <= CORTE_ALTO:
        return "04_alto"
    else:
        return "05_ultra_alto"


df_estratos = features_candidatos.copy()

df_estratos["estrato_patrimonial"] = df_estratos[patrimonio_col].apply(
    criar_estrato_patrimonial
)

tabela_estratos_adotados = (
    df_estratos
    .groupby("estrato_patrimonial")
    .agg(
        qtd_candidatos=("SQ_CANDIDATO", "count"),
        patrimonio_min=(patrimonio_col, "min"),
        patrimonio_max=(patrimonio_col, "max"),
        patrimonio_medio=(patrimonio_col, "mean"),
        qtd_bens_media=("qtd_bens", "mean"),
        qtd_macros_media=("qtd_macros_presentes", "mean"),
        concentracao_media=("indice_concentracao_macro", "mean"),
    )
    .reset_index()
)

tabela_estratos_adotados["perc_base"] = (
    tabela_estratos_adotados["qtd_candidatos"] / len(df_estratos)
)

tabela_estratos_adotados = tabela_estratos_adotados.sort_values(
    "patrimonio_min"
)

tabela_estratos_adotados

,estrato_patrimonial,qtd_candidatos,patrimonio_min,patrimonio_max,patrimonio_medio,qtd_bens_media,qtd_macros_media,concentracao_media,perc_base
0,01_baixo,5365,0.01,1.000000e+05,3.833371e+04,1.777260,1.326375,0.918626,0.294473
1,02_medio_baixo,4186,100021.05,3.000000e+05,1.920508e+05,3.098423,2.031295,0.760987,0.229760
2,03_medio_alto,4069,300005.52,7.500000e+05,4.879935e+05,4.768493,2.625461,0.694278,0.223338
3,04_alto,4410,750149.76,1.300000e+07,2.188159e+06,10.125850,3.552154,0.616115,0.242055
4,05_ultra_alto,189,13114000.00,1.267951e+09,5.839267e+07,31.354497,5.031746,0.590253,0.010374


In [45]:
# CRIAÇÃO DO ESTRATO PATRIMONIAL NO DATAFRAME PRINCIPAL

def criar_estrato_patrimonial(valor):
    if valor <= CORTE_BAIXO:
        return "01_baixo"
    elif valor <= CORTE_MEDIO_BAIXO:
        return "02_medio_baixo"
    elif valor <= CORTE_MEDIO_ALTO:
        return "03_medio_alto"
    elif valor <= CORTE_ALTO:
        return "04_alto"
    else:
        return "05_ultra_alto"


features_candidatos["estrato_patrimonial"] = features_candidatos[
    "patrimonio_total"
].apply(criar_estrato_patrimonial)

features_candidatos["estrato_patrimonial"].value_counts().sort_index()

estrato_patrimonial
01_baixo          5365
02_medio_baixo    4186
03_medio_alto     4069
04_alto           4410
05_ultra_alto      189
Name: count, dtype: int64

## 4. Transformações para clusterização

Após a estratificação patrimonial, foram aplicadas transformações distintas conforme o tipo de estrato analisado. Essa decisão foi tomada porque os quatro primeiros estratos e o estrato ultra_alto têm funções analíticas diferentes no pipeline.

Nos estratos de patrimônio baixo, médio-baixo, médio-alto e alto, a matriz de entrada do K-Means é composta pelos valores absolutos declarados em cada macro categoria patrimonial. Antes da clusterização, esses valores são transformados por `log1p`, isto é:

$$
x' = \log(1 + x)
$$

Essa transformação reduz a assimetria dos valores monetários e diminui o peso de diferenças absolutas muito grandes, sem eliminar completamente a informação de magnitude patrimonial. Assim, candidatos com estruturas semelhantes, mas valores muito diferentes dentro do mesmo estrato, não dominam excessivamente o cálculo das distâncias.

No estrato ultra_alto, formado por candidatos com patrimônio acima de R$ 13 milhões, a clusterização utiliza os percentuais de composição patrimonial por macro categoria. Sobre esses percentuais é aplicada a transformação de Hellinger:

$$
x' = \sqrt{x}
$$

Nesse caso, a magnitude patrimonial já foi isolada pela própria estratificação. Por isso, o objetivo do clustering passa a ser comparar a composição relativa do patrimônio, e não o volume absoluto declarado. A transformação de Hellinger é adequada para dados composicionais, pois permite comparar distribuições de percentuais preservando a estrutura relativa entre categorias e reduzindo distorções geradas pela soma constante dos percentuais.

In [46]:
colunas_valores = [
    col for col in features_candidatos.columns
    if col.startswith("valor_")
]

colunas_percentuais = [
    col for col in features_candidatos.columns
    if col.startswith("perc_")
]

In [47]:
# 4. APLICAÇÃO DAS TRANSFORMAÇÕES PARA CLUSTERIZAÇÃO
def preparar_matriz_clustering(df_estrato, estrato):
    """
    Prepara a matriz de entrada do K-Means para um estrato patrimonial.

    Para os quatro primeiros estratos, utiliza log1p dos valores absolutos
    por macro categoria.

    Para o estrato ultra_alto, utiliza a transformação de Hellinger, isto é,
    a raiz quadrada dos percentuais por macro categoria.
    """

    if estrato == "05_ultra_alto":
        X = np.sqrt(
            df_estrato[colunas_percentuais]
            .fillna(0)
            .clip(lower=0)
            .to_numpy()
        )

        tipo_matriz = "hellinger_percentuais"

    else:
        X = np.log1p(
            df_estrato[colunas_valores]
            .fillna(0)
            .clip(lower=0)
            .to_numpy()
        )

        tipo_matriz = "log_valores"

    return X, tipo_matriz

## 5. Clusterização via K-Means

Após a definição dos estratos patrimoniais, das transformações e do número de clusters de cada faixa, o K-Means foi aplicado separadamente dentro de cada estrato. Essa decisão mantém a comparação entre candidatos de escalas patrimoniais semelhantes e reduz o efeito da assimetria da distribuição de patrimônio.

Os valores de `k` utilizados foram definidos previamente a partir da análise exploratória das métricas internas e da interpretação dos perfis resultantes. O parâmetro `random_state` foi fixado para garantir reprodutibilidade, enquanto `n_init` foi mantido elevado para reduzir a dependência do algoritmo em relação à inicialização aleatória dos centróides.

Para cada estrato, foram armazenados os rótulos dos clusters, as métricas internas de qualidade e estatísticas descritivas dos grupos formados. As métricas são utilizadas como apoio à avaliação dos agrupamentos, mas a escolha final também considera equilíbrio entre tamanhos de clusters e interpretabilidade substantiva dos perfis patrimoniais.

In [48]:
# 5. APLICAÇÃO DO K-MEANS
#
# Para cada estrato:
#   1. filtra os candidatos do estrato;
#   2. prepara a matriz de clustering com a transformação adequada;
#   3. aplica K-Means com o k definido previamente;
#   4. calcula métricas de qualidade;
#   5. armazena os rótulos finais dos candidatos.

from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
import numpy as np

resultados_metricas = []
lista_rotulos = []
modelos_kmeans = {}

for estrato in ESTRATOS_ORDEM:
    df_estrato = features_candidatos[
        features_candidatos["estrato_patrimonial"] == estrato
    ].copy()

    k = K_POR_ESTRATO[estrato]

    X_cluster, tipo_matriz = preparar_matriz_clustering(
        df_estrato=df_estrato,
        estrato=estrato
    )

    modelo = KMeans(
        n_clusters=k,
        random_state=RANDOM_STATE,
        n_init=N_INIT_KMEANS
    )

    labels = modelo.fit_predict(X_cluster)

    df_rotulos_estrato = df_estrato[[
        "SQ_CANDIDATO",
        "estrato_patrimonial",
        "patrimonio_total",
        "qtd_bens",
        "qtd_macros_presentes",
        "indice_concentracao_macro"
    ]].copy()

    df_rotulos_estrato["k"] = k
    df_rotulos_estrato["cluster"] = labels
    df_rotulos_estrato["cluster_global"] = (
        df_rotulos_estrato["estrato_patrimonial"]
        + "_c"
        + df_rotulos_estrato["cluster"].astype(str)
    )
    df_rotulos_estrato["tipo_matriz"] = tipo_matriz

    contagem_clusters = pd.Series(labels).value_counts().sort_index()

    metricas = {
        "estrato_patrimonial": estrato,
        "k": k,
        "tipo_matriz": tipo_matriz,
        "n_candidatos": len(df_estrato),
        "inercia": modelo.inertia_,
        "silhouette": silhouette_score(X_cluster, labels),
        "davies_bouldin": davies_bouldin_score(X_cluster, labels),
        "calinski_harabasz": calinski_harabasz_score(X_cluster, labels),
        "menor_cluster": contagem_clusters.min(),
        "maior_cluster": contagem_clusters.max(),
        "perc_menor_cluster": contagem_clusters.min() / len(df_estrato),
        "perc_maior_cluster": contagem_clusters.max() / len(df_estrato),
    }

    resultados_metricas.append(metricas)
    lista_rotulos.append(df_rotulos_estrato)
    modelos_kmeans[estrato] = modelo


metricas_clusters_finais = pd.DataFrame(resultados_metricas)
rotulos_clusters_finais = pd.concat(lista_rotulos, ignore_index=True)

metricas_clusters_finais

,estrato_patrimonial,k,tipo_matriz,n_candidatos,inercia,silhouette,davies_bouldin,calinski_harabasz,menor_cluster,maior_cluster,perc_menor_cluster,perc_maior_cluster
0,01_baixo,8,log_valores,5365,85299.898766,0.646196,0.888624,3095.548029,258,2191,0.048089,0.408388
1,02_medio_baixo,8,log_valores,4186,156360.543798,0.526478,1.093072,1266.839269,217,1154,0.051839,0.275681
2,03_medio_alto,7,log_valores,4069,237603.042504,0.415293,1.321295,901.862263,392,1152,0.096338,0.283116
3,04_alto,8,log_valores,4410,431552.616424,0.263910,1.508895,708.334696,410,677,0.092971,0.153515
4,05_ultra_alto,5,hellinger_percentuais,189,34.335668,0.346242,1.138793,81.209547,23,57,0.121693,0.301587


In [49]:
rotulos_clusters_finais.head()

,SQ_CANDIDATO,estrato_patrimonial,patrimonio_total,qtd_bens,qtd_macros_presentes,indice_concentracao_macro,k,cluster,cluster_global,tipo_matriz
0,10001595335,01_baixo,60000.0,2,1,1.0,8,7,01_baixo_c7,log_valores
1,10001595340,01_baixo,32000.0,2,1,1.0,8,7,01_baixo_c7,log_valores
2,10001595351,01_baixo,100000.0,1,1,1.0,8,3,01_baixo_c3,log_valores
3,10001595354,01_baixo,10000.0,1,1,1.0,8,7,01_baixo_c7,log_valores
4,10001595357,01_baixo,60000.0,2,1,1.0,8,7,01_baixo_c7,log_valores


In [50]:
distribuicao_clusters_finais = (
    rotulos_clusters_finais
    .groupby(["estrato_patrimonial", "cluster", "cluster_global"])
    .agg(
        qtd_candidatos=("SQ_CANDIDATO", "count"),
        patrimonio_medio=("patrimonio_total", "mean"),
        patrimonio_mediano=("patrimonio_total", "median"),
        qtd_bens_media=("qtd_bens", "mean"),
        qtd_macros_media=("qtd_macros_presentes", "mean"),
        concentracao_media=("indice_concentracao_macro", "mean"),
    )
    .reset_index()
)

distribuicao_clusters_finais["perc_estrato"] = (
    distribuicao_clusters_finais["qtd_candidatos"]
    / distribuicao_clusters_finais.groupby("estrato_patrimonial")["qtd_candidatos"].transform("sum")
)

distribuicao_clusters_finais

,estrato_patrimonial,cluster,cluster_global,qtd_candidatos,patrimonio_medio,patrimonio_mediano,qtd_bens_media,qtd_macros_media,concentracao_media,perc_estrato
0,01_baixo,0,01_baixo_c0,411,3.603688e+04,2.203609e+04,1.929440,1.394161,0.890767,0.076608
1,01_baixo,1,01_baixo_c1,690,1.680048e+04,6.548510e+03,2.110145,1.140580,0.971064,0.128611
2,01_baixo,2,01_baixo_c2,258,2.298601e+04,7.110000e+03,1.364341,1.100775,0.988504,0.048089
3,01_baixo,3,01_baixo_c3,718,6.034895e+04,6.000000e+04,1.511142,1.245125,0.949298,0.133830
4,01_baixo,4,01_baixo_c4,364,2.223065e+04,1.000000e+04,1.585165,1.401099,0.891297,0.067847
5,01_baixo,5,01_baixo_c5,306,7.024517e+04,7.400000e+04,2.826797,2.346405,0.615004,0.057036
6,01_baixo,6,01_baixo_c6,427,5.066418e+04,4.904918e+04,3.515222,2.257611,0.704600,0.079590
7,01_baixo,7,01_baixo_c7,2191,3.595407e+04,3.000000e+04,1.326335,1.089000,0.977714,0.408388
8,02_medio_baixo,0,02_medio_baixo_c0,360,1.789275e+05,1.700000e+05,4.919444,2.391667,0.654360,0.086001
9,02_medio_baixo,1,02_medio_baixo_c1,1154,1.959964e+05,1.950000e+05,2.751300,2.169844,0.680863,0.275681


## 6. Algoritmo de Nomeação dos clusters

Após a geração dos clusters , foi criada uma etapa de nomeação automática dos perfis. O objetivo dessa etapa é substituir identificadores técnicos, como `cluster_0_baixo` ou `cluster_1_alto`, por nomes interpretáveis baseados na composição patrimonial média de cada agrupamento.

A regra de nomeação utiliza as macro categorias patrimoniais com maior participação percentual média em cada cluster. Quando uma única macro representa pelo menos 70% da composição média do grupo, o cluster é classificado como um perfil concentrado. Quando as duas principais macros somam pelo menos 70%, o nome combina essas duas categorias. Nos demais casos, o cluster é classificado como diversificado, tomando como referência a macro predominante.

In [51]:

# 6. NOMEAÇÃO AUTOMÁTICA DOS CLUSTERS
# Objetivo:
# Regra geral:
# - se uma macro representa pelo menos 70% do cluster:
#       "Estrato - macro concentrado"
# - se as duas principais macros somam pelo menos 70%:
#       "Estrato - macro 1 + macro 2"
# - caso contrário:
#       "Estrato - macro 1 diversificado"
#
# Observação:
# O número do cluster do K-Means é mantido apenas como identificador técnico.
# O nome interpretativo é derivado da composição média do cluster.

LIMIAR_CONCENTRADO = 0.70
LIMIAR_DUAS_MACROS = 0.70

NOMES_ESTRATOS = {
    "01_baixo": "Baixo patrimônio",
    "02_medio_baixo": "Patrimônio médio-baixo",
    "03_medio_alto": "Patrimônio médio-alto",
    "04_alto": "Alto patrimônio",
    "05_ultra_alto": "Patrimônio ultra-alto",
}

colunas_percentuais_existentes = [
    col for col in colunas_percentuais
    if col in features_candidatos.columns
]

In [52]:
def obter_top_macros(row, colunas_percentuais):
    """
    Retorna as três principais macros patrimoniais de uma linha,
    ordenadas pelo percentual médio no cluster.
    """

    valores = row[colunas_percentuais].sort_values(ascending=False)

    macro_1_col = valores.index[0]
    macro_1_pct = valores.iloc[0]

    macro_2_col = valores.index[1] if len(valores) > 1 else None
    macro_2_pct = valores.iloc[1] if len(valores) > 1 else 0

    macro_3_col = valores.index[2] if len(valores) > 2 else None
    macro_3_pct = valores.iloc[2] if len(valores) > 2 else 0

    return pd.Series({
        "macro_1_col": macro_1_col,
        "macro_1_pct": macro_1_pct,
        "macro_2_col": macro_2_col,
        "macro_2_pct": macro_2_pct,
        "macro_3_col": macro_3_col,
        "macro_3_pct": macro_3_pct,
    })


def montar_nome_cluster(row):
    """
    Monta nome interpretativo simples para o cluster.
    """

    nome_estrato = NOMES_ESTRATOS.get(
        row["estrato_patrimonial"],
        row["estrato_patrimonial"]
    )

    macro_1 = NOMES_MACROS.get(row["macro_1_col"], row["macro_1_col"])
    macro_2 = NOMES_MACROS.get(row["macro_2_col"], row["macro_2_col"])

    macro_1_pct = row["macro_1_pct"]
    macro_2_pct = row["macro_2_pct"]

    if macro_1_pct >= LIMIAR_CONCENTRADO:
        nome_base = f"{macro_1} concentrado"

    elif macro_1_pct + macro_2_pct >= LIMIAR_DUAS_MACROS:
        nome_base = f"{macro_1} + {macro_2}"

    else:
        nome_base = f"{macro_1} diversificado"

    return f"{nome_estrato} - {nome_base}"

In [53]:
base_nomeacao_clusters = rotulos_clusters_finais.merge(
    features_candidatos[
        ["SQ_CANDIDATO"] + colunas_percentuais_existentes
    ],
    on="SQ_CANDIDATO",
    how="left"
)

# Calcula a composição média de cada cluster

perfil_composicao_clusters = (
    base_nomeacao_clusters
    .groupby(["estrato_patrimonial", "cluster", "cluster_global"])
    .agg(
        qtd_candidatos=("SQ_CANDIDATO", "count"),
        patrimonio_medio=("patrimonio_total", "mean"),
        patrimonio_mediano=("patrimonio_total", "median"),
        qtd_bens_media=("qtd_bens", "mean"),
        qtd_macros_media=("qtd_macros_presentes", "mean"),
        concentracao_media=("indice_concentracao_macro", "mean"),
        **{col: (col, "mean") for col in colunas_percentuais_existentes}
    )
    .reset_index()
)

# Identifica macro dominante, segunda macro e terceira macro

top_macros = perfil_composicao_clusters.apply(
    obter_top_macros,
    axis=1,
    colunas_percentuais=colunas_percentuais_existentes
)

perfil_composicao_clusters = pd.concat(
    [perfil_composicao_clusters, top_macros],
    axis=1
)

# Gera nome automático

perfil_composicao_clusters["perfil_automatico"] = (
    perfil_composicao_clusters.apply(montar_nome_cluster, axis=1)
)

In [54]:
# DESAMBIGUAÇÃO SIMPLES DE NOMES REPETIDOS

# Se dois clusters do mesmo estrato receberem o mesmo nome,
# adicionamos a segunda macro mais relevante ao nome.

def desambiguar_nome(row):
    macro_2 = NOMES_MACROS.get(row["macro_2_col"], row["macro_2_col"])
    macro_3 = NOMES_MACROS.get(row["macro_3_col"], row["macro_3_col"])

    nome = row["perfil_automatico"]

    if not row["nome_repetido"]:
        return nome

    if pd.notna(macro_2):
        nome = f"{nome} com {macro_2}"

    if pd.notna(macro_3):
        nome = f"{nome} e {macro_3}"

    return nome


perfil_composicao_clusters["nome_repetido"] = (
    perfil_composicao_clusters
    .groupby(["estrato_patrimonial", "perfil_automatico"])
    ["cluster"]
    .transform("count")
    .gt(1)
)

perfil_composicao_clusters["perfil_nomeado"] = (
    perfil_composicao_clusters.apply(desambiguar_nome, axis=1)
)

In [55]:
# APLICAÇÃO DOS NOMES AOS CANDIDATOS

mapa_nomes_clusters = perfil_composicao_clusters.set_index(
    "cluster_global"
)["perfil_nomeado"].to_dict()

rotulos_clusters_nomeados = rotulos_clusters_finais.copy()

rotulos_clusters_nomeados["perfil_cluster"] = (
    rotulos_clusters_nomeados["cluster_global"].map(mapa_nomes_clusters)
)

rotulos_clusters_nomeados.head()

,SQ_CANDIDATO,estrato_patrimonial,patrimonio_total,qtd_bens,qtd_macros_presentes,indice_concentracao_macro,k,cluster,cluster_global,tipo_matriz,perfil_cluster
0,10001595335,01_baixo,60000.0,2,1,1.0,8,7,01_baixo_c7,log_valores,Baixo patrimônio - veicular concentrado com ru...
1,10001595340,01_baixo,32000.0,2,1,1.0,8,7,01_baixo_c7,log_valores,Baixo patrimônio - veicular concentrado com ru...
2,10001595351,01_baixo,100000.0,1,1,1.0,8,3,01_baixo_c3,log_valores,Baixo patrimônio - imobiliário concentrado
3,10001595354,01_baixo,10000.0,1,1,1.0,8,7,01_baixo_c7,log_valores,Baixo patrimônio - veicular concentrado com ru...
4,10001595357,01_baixo,60000.0,2,1,1.0,8,7,01_baixo_c7,log_valores,Baixo patrimônio - veicular concentrado com ru...


In [56]:
# TABELA DE AUDITORIA DOS NOMES GERADOS
nomes_clusters_automaticos = perfil_composicao_clusters[[
    "estrato_patrimonial",
    "cluster",
    "cluster_global",
    "perfil_nomeado",
    "qtd_candidatos",
    "patrimonio_medio",
    "patrimonio_mediano",
    "qtd_bens_media",
    "qtd_macros_media",
    "concentracao_media",
    "macro_1_col",
    "macro_1_pct",
    "macro_2_col",
    "macro_2_pct",
    "macro_3_col",
    "macro_3_pct",
    "nome_repetido",
]].copy()

nomes_clusters_automaticos["macro_1"] = nomes_clusters_automaticos["macro_1_col"].map(NOMES_MACROS)
nomes_clusters_automaticos["macro_2"] = nomes_clusters_automaticos["macro_2_col"].map(NOMES_MACROS)
nomes_clusters_automaticos["macro_3"] = nomes_clusters_automaticos["macro_3_col"].map(NOMES_MACROS)

nomes_clusters_automaticos = nomes_clusters_automaticos.sort_values(
    ["estrato_patrimonial", "cluster"]
).reset_index(drop=True)

nomes_clusters_automaticos

,estrato_patrimonial,cluster,cluster_global,perfil_nomeado,qtd_candidatos,patrimonio_medio,patrimonio_mediano,qtd_bens_media,qtd_macros_media,concentracao_media,macro_1_col,macro_1_pct,macro_2_col,macro_2_pct,macro_3_col,macro_3_pct,nome_repetido,macro_1,macro_2,macro_3
0,01_baixo,0,01_baixo_c0,Baixo patrimônio - societário concentrado,411,3.603688e+04,2.203609e+04,1.929440,1.394161,0.890767,perc_participacoes_societarias,0.889662,perc_ativos_financeiros,0.057458,perc_veiculos,0.030602,False,societário,financeiro,veicular
1,01_baixo,1,01_baixo_c1,Baixo patrimônio - financeiro concentrado,690,1.680048e+04,6.548510e+03,2.110145,1.140580,0.971064,perc_ativos_financeiros,0.957912,perc_creditos_direitos,0.014686,perc_outros,0.009046,False,financeiro,créditos e direitos,outros ativos
2,01_baixo,2,01_baixo_c2,Baixo patrimônio - outros ativos diversificado,258,2.298601e+04,7.110000e+03,1.364341,1.100775,0.988504,perc_outros,0.549902,perc_rural_agropecuario,0.144765,perc_ativos_financeiros,0.130343,False,outros ativos,rural/agropecuário,financeiro
3,01_baixo,3,01_baixo_c3,Baixo patrimônio - imobiliário concentrado,718,6.034895e+04,6.000000e+04,1.511142,1.245125,0.949298,perc_imoveis,0.958458,perc_ativos_financeiros,0.014860,perc_participacoes_societarias,0.011887,False,imobiliário,financeiro,societário
4,01_baixo,4,01_baixo_c4,Baixo patrimônio - dinheiro em espécie concent...,364,2.223065e+04,1.000000e+04,1.585165,1.401099,0.891297,perc_dinheiro_especie,0.897171,perc_ativos_financeiros,0.045303,perc_veiculos,0.027817,False,dinheiro em espécie,financeiro,veicular
5,01_baixo,5,01_baixo_c5,Baixo patrimônio - imobiliário + veicular,306,7.024517e+04,7.400000e+04,2.826797,2.346405,0.615004,perc_imoveis,0.604695,perc_veiculos,0.350083,perc_participacoes_societarias,0.016399,False,imobiliário,veicular,societário
6,01_baixo,6,01_baixo_c6,Baixo patrimônio - veicular concentrado com fi...,427,5.066418e+04,4.904918e+04,3.515222,2.257611,0.704600,perc_veiculos,0.737090,perc_ativos_financeiros,0.216307,perc_participacoes_societarias,0.016384,True,veicular,financeiro,societário
7,01_baixo,7,01_baixo_c7,Baixo patrimônio - veicular concentrado com ru...,2191,3.595407e+04,3.000000e+04,1.326335,1.089000,0.977714,perc_veiculos,0.979250,perc_rural_agropecuario,0.006793,perc_dinheiro_especie,0.004207,True,veicular,rural/agropecuário,dinheiro em espécie
8,02_medio_baixo,0,02_medio_baixo_c0,Patrimônio médio-baixo - veicular + financeiro,360,1.789275e+05,1.700000e+05,4.919444,2.391667,0.654360,perc_veiculos,0.415215,perc_ativos_financeiros,0.411949,perc_dinheiro_especie,0.066745,False,veicular,financeiro,dinheiro em espécie
9,02_medio_baixo,1,02_medio_baixo_c1,Patrimônio médio-baixo - imobiliário concentra...,1154,1.959964e+05,1.950000e+05,2.751300,2.169844,0.680863,perc_imoveis,0.744614,perc_veiculos,0.232092,perc_participacoes_societarias,0.011482,True,imobiliário,veicular,societário


## 7. Aplicação dos perfis à matriz original e exportação de resultados finais

Após a definição dos clusters e a nomeação automática dos perfis patrimoniais, os resultados foram incorporados à matriz final de features por candidato. Essa etapa gera uma base consolidada contendo as variáveis patrimoniais originais, o estrato patrimonial atribuído e o perfil de cluster final.

A coluna `estrato` identifica a faixa patrimonial do candidato, enquanto a coluna `perfil_cluster` contém o nome interpretável do agrupamento ao qual ele foi associado. 

O arquivo final é exportado como `features_patrimoniais_por_candidato_cluster.csv` e pode ser utilizado em análises posteriores, visualizações ou integração com outras bases do projeto.

In [57]:
# 7. APLICAÇÃO DOS RESULTADOS À MATRIZ ORIGINAL DE FEATURES

features_patrimoniais_por_candidato_cluster = features_candidatos.copy()

colunas_resultado_cluster = rotulos_clusters_nomeados[[
    "SQ_CANDIDATO",
    "estrato_patrimonial",
    "perfil_cluster"
]].copy()

colunas_resultado_cluster = colunas_resultado_cluster.rename(
    columns={
        "estrato_patrimonial": "estrato"
    }
)

features_patrimoniais_por_candidato_cluster = (
    features_patrimoniais_por_candidato_cluster
    .drop(columns=["estrato_patrimonial"], errors="ignore")
    .merge(
        colunas_resultado_cluster,
        on="SQ_CANDIDATO",
        how="left"
    )
)

features_patrimoniais_por_candidato_cluster.head()

,SQ_CANDIDATO,valor_ativos_financeiros,valor_bens_luxo_colecao,valor_creditos_direitos,valor_dinheiro_especie,valor_direitos_intangiveis,valor_imoveis,valor_outros,valor_outros_atividade_profissional,valor_participacoes_societarias,...,perc_outros_atividade_profissional,perc_participacoes_societarias,perc_rural_agropecuario,perc_veiculos,qtd_bens,qtd_macros_presentes,indice_concentracao_macro,patrimonio_total,estrato,perfil_cluster
0,10001595335,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.000000,2,1,1.000000,60000.0,01_baixo,Baixo patrimônio - veicular concentrado com ru...
1,10001595336,0.0,0.0,0.0,0.0,0.0,400000.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.221638,3,2,0.654970,513900.0,03_medio_alto,Patrimônio médio-alto - imobiliário concentrad...
2,10001595338,0.0,0.0,0.0,0.0,0.0,300000.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.038462,2,2,0.926036,312000.0,03_medio_alto,Patrimônio médio-alto - imobiliário concentrad...
3,10001595339,0.0,0.0,0.0,0.0,0.0,250000.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.285714,2,2,0.591837,350000.0,03_medio_alto,Patrimônio médio-alto - imobiliário concentrad...
4,10001595340,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.000000,2,1,1.000000,32000.0,01_baixo,Baixo patrimônio - veicular concentrado com ru...


In [58]:
# CHECAGENS DA APLICAÇÃO DOS CLUSTERS
features_patrimoniais_por_candidato_cluster[[
    "estrato",
    "perfil_cluster"
]].value_counts().reset_index(name="qtd_candidatos")

,estrato,perfil_cluster,qtd_candidatos
0,01_baixo,Baixo patrimônio - veicular concentrado com ru...,2191
1,02_medio_baixo,Patrimônio médio-baixo - imobiliário concentra...,1154
2,03_medio_alto,Patrimônio médio-alto - imobiliário concentrad...,1152
3,04_alto,Alto patrimônio - imobiliário + societário com...,1060
4,02_medio_baixo,Patrimônio médio-baixo - imobiliário concentra...,1033
5,01_baixo,Baixo patrimônio - imobiliário concentrado,718
6,01_baixo,Baixo patrimônio - financeiro concentrado,690
7,04_alto,Alto patrimônio - imobiliário + financeiro,677
8,03_medio_alto,Patrimônio médio-alto - imobiliário concentrad...,670
9,04_alto,Alto patrimônio - imobiliário + rural/agropecu...,636


In [59]:
# EXPORTAÇÃO DO DATASET FINAL 
features_patrimoniais_por_candidato_cluster.to_csv(
    "features_patrimoniais_por_candidato_cluster.csv",
    sep=";",
    index=False,
    encoding="utf-8-sig"
)

In [60]:
from pathlib import Path

def salvar_auditoria_clusters(
    metricas_clusters_finais,
    rotulos_clusters_finais,
    distribuicao_clusters_finais,
    nomes_clusters_automaticos=None,
    rotulos_clusters_nomeados=None,
    gerar_auditoria=False,
    pasta_saida="auditoria_clusters"
):
    """
    Salva os arquivos de auditoria dos clusters, caso gerar_auditoria=True.

    Arquivos gerados:
        - metricas_clusters_finais.csv
        - rotulos_clusters_finais.csv
        - distribuicao_clusters_finais.csv

    Arquivos opcionais:
        - nomes_clusters_automaticos.csv
        - rotulos_clusters_nomeados.csv
    """

    if not gerar_auditoria:
        print("Auditoria de clusters desativada. Nenhum arquivo foi salvo.")
        return None

    pasta_saida = Path(pasta_saida)
    pasta_saida.mkdir(parents=True, exist_ok=True)

    caminhos = {
        "metricas": pasta_saida / "metricas_clusters_finais.csv",
        "rotulos": pasta_saida / "rotulos_clusters_finais.csv",
        "distribuicao": pasta_saida / "distribuicao_clusters_finais.csv",
    }

    metricas_clusters_finais.to_csv(
        caminhos["metricas"],
        sep=";",
        index=False,
        encoding="utf-8-sig"
    )

    rotulos_clusters_finais.to_csv(
        caminhos["rotulos"],
        sep=";",
        index=False,
        encoding="utf-8-sig"
    )

    distribuicao_clusters_finais.to_csv(
        caminhos["distribuicao"],
        sep=";",
        index=False,
        encoding="utf-8-sig"
    )

    if nomes_clusters_automaticos is not None:
        caminhos["nomes_clusters"] = pasta_saida / "nomes_clusters_automaticos.csv"

        nomes_clusters_automaticos.to_csv(
            caminhos["nomes_clusters"],
            sep=";",
            index=False,
            encoding="utf-8-sig"
        )

    if rotulos_clusters_nomeados is not None:
        caminhos["rotulos_nomeados"] = pasta_saida / "rotulos_clusters_nomeados.csv"

        rotulos_clusters_nomeados.to_csv(
            caminhos["rotulos_nomeados"],
            sep=";",
            index=False,
            encoding="utf-8-sig"
        )

    print("Arquivos de auditoria salvos em:", pasta_saida.resolve())

    return caminhos

In [61]:
caminhos_auditoria_clusters = salvar_auditoria_clusters(
    metricas_clusters_finais=metricas_clusters_finais,
    rotulos_clusters_finais=rotulos_clusters_finais,
    distribuicao_clusters_finais=distribuicao_clusters_finais,
    nomes_clusters_automaticos=nomes_clusters_automaticos,
    rotulos_clusters_nomeados=rotulos_clusters_nomeados,
    gerar_auditoria=GERAR_AUDITORIA_CLUSTERS,
    pasta_saida=PASTA_AUDITORIA_CLUSTERS
)

Auditoria de clusters desativada. Nenhum arquivo foi salvo.


## Seção 8: busca exploratória de K

Esta seção permite reavaliar diferentes valores de `k` para cada estrato patrimonial. Ela só é executada quando o parâmetro `RODAR_BUSCAS_METRICAS` está definido como `True`.

Para cada estrato e cada valor de `k` definido em `RANGES_K`, o K-Means é ajustado com os mesmos parâmetros do modelo final, incluindo `random_state` e `n_init`. Em seguida, são calculadas métricas internas de qualidade, como silhouette, Davies-Bouldin e Calinski-Harabasz, além de informações sobre o tamanho do menor e do maior cluster.

Além das métricas globais, a seção também gera estatísticas descritivas dos clusters formados, incluindo patrimônio médio e mediano, quantidade média de bens, quantidade média de macro categorias, índice médio de concentração e macro categorias dominantes. Essas informações auxiliam a avaliar não apenas a separação geométrica dos clusters, mas também sua interpretabilidade substantiva.

A busca exploratória é utilizada como apoio metodológico. A escolha final de `k` não depende exclusivamente das métricas internas, mas também do equilíbrio entre tamanho dos grupos, estabilidade e clareza dos perfis patrimoniais encontrados.


In [62]:
# SEÇÃO OPCIONAL: BUSCA EXPLORATÓRIA DE K POR ESTRATO
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score

def identificar_macros_dominantes_linha(row, colunas_percentuais):
    """
    Identifica as três macros com maior percentual médio em uma linha.
    Usada para resumir a composição média de cada cluster.
    """

    valores = row[colunas_percentuais].sort_values(ascending=False)

    macro_1_col = valores.index[0]
    macro_1_pct = valores.iloc[0]

    macro_2_col = valores.index[1] if len(valores) > 1 else None
    macro_2_pct = valores.iloc[1] if len(valores) > 1 else 0

    macro_3_col = valores.index[2] if len(valores) > 2 else None
    macro_3_pct = valores.iloc[2] if len(valores) > 2 else 0

    return pd.Series({
        "macro_1_col": macro_1_col,
        "macro_1_pct": macro_1_pct,
        "macro_2_col": macro_2_col,
        "macro_2_pct": macro_2_pct,
        "macro_3_col": macro_3_col,
        "macro_3_pct": macro_3_pct,
    })

In [63]:
def avaliar_k_por_estrato(
    features_candidatos,
    estratos_ordem,
    ranges_k,
    colunas_percentuais,
    random_state=42,
    n_init=50
):
    """
    Avalia diferentes valores de k para cada estrato patrimonial.

    Retorna:
        - metricas_busca_k:
            tabela agregada por estrato e k.

        - perfis_busca_k:
            tabela detalhada por estrato, k e cluster.
    """

    lista_metricas = []
    lista_perfis = []

    colunas_percentuais_existentes = [
        col for col in colunas_percentuais
        if col in features_candidatos.columns
    ]

    for estrato in estratos_ordem:

        df_estrato = features_candidatos[
            features_candidatos["estrato_patrimonial"] == estrato
        ].copy()

        if len(df_estrato) == 0:
            print(f"Estrato sem candidatos: {estrato}")
            continue

        for k in ranges_k[estrato]:

            if k >= len(df_estrato):
                print(f"Pulando {estrato}, k={k}: k maior ou igual ao número de candidatos.")
                continue

            X_cluster, tipo_matriz = preparar_matriz_clustering(
                df_estrato=df_estrato,
                estrato=estrato
            )

            modelo = KMeans(
                n_clusters=k,
                random_state=random_state,
                n_init=n_init
            )

            labels = modelo.fit_predict(X_cluster)

            df_tmp = df_estrato.copy()
            df_tmp["cluster"] = labels

            contagem_clusters = (
                df_tmp["cluster"]
                .value_counts()
                .sort_index()
            )

            menor_cluster = contagem_clusters.min()
            maior_cluster = contagem_clusters.max()

            cluster_menor_id = contagem_clusters.idxmin()
            cluster_maior_id = contagem_clusters.idxmax()

            # ------------------------------------------------
            # Perfil por cluster
            # ------------------------------------------------

            perfil_clusters = (
                df_tmp
                .groupby("cluster")
                .agg(
                    qtd_candidatos=("SQ_CANDIDATO", "count"),

                    patrimonio_min=("patrimonio_total", "min"),
                    patrimonio_max=("patrimonio_total", "max"),
                    patrimonio_medio=("patrimonio_total", "mean"),
                    patrimonio_mediano=("patrimonio_total", "median"),

                    qtd_bens_media=("qtd_bens", "mean"),
                    qtd_bens_mediana=("qtd_bens", "median"),

                    qtd_macros_media=("qtd_macros_presentes", "mean"),
                    qtd_macros_mediana=("qtd_macros_presentes", "median"),

                    concentracao_media=("indice_concentracao_macro", "mean"),
                    concentracao_mediana=("indice_concentracao_macro", "median"),

                    **{col: (col, "mean") for col in colunas_percentuais_existentes}
                )
                .reset_index()
            )

            perfil_clusters["estrato_patrimonial"] = estrato
            perfil_clusters["k"] = k
            perfil_clusters["tipo_matriz"] = tipo_matriz
            perfil_clusters["random_state"] = random_state
            perfil_clusters["n_init"] = n_init
            perfil_clusters["perc_estrato"] = (
                perfil_clusters["qtd_candidatos"] / len(df_estrato)
            )

            top_macros = perfil_clusters.apply(
                identificar_macros_dominantes_linha,
                axis=1,
                colunas_percentuais=colunas_percentuais_existentes
            )

            perfil_clusters = pd.concat(
                [perfil_clusters, top_macros],
                axis=1
            )

            perfil_clusters["macro_1"] = perfil_clusters["macro_1_col"].map(NOMES_MACROS)
            perfil_clusters["macro_2"] = perfil_clusters["macro_2_col"].map(NOMES_MACROS)
            perfil_clusters["macro_3"] = perfil_clusters["macro_3_col"].map(NOMES_MACROS)

            # ------------------------------------------------
            # Métricas agregadas do k
            # ------------------------------------------------

            metricas_k = {
                "estrato_patrimonial": estrato,
                "k": k,
                "tipo_matriz": tipo_matriz,
                "random_state": random_state,
                "n_init": n_init,

                "n_candidatos": len(df_estrato),
                "inercia": modelo.inertia_,
                "silhouette": silhouette_score(X_cluster, labels),
                "davies_bouldin": davies_bouldin_score(X_cluster, labels),
                "calinski_harabasz": calinski_harabasz_score(X_cluster, labels),

                "menor_cluster": menor_cluster,
                "maior_cluster": maior_cluster,
                "perc_menor_cluster": menor_cluster / len(df_estrato),
                "perc_maior_cluster": maior_cluster / len(df_estrato),
                "cluster_menor_id": cluster_menor_id,
                "cluster_maior_id": cluster_maior_id,

                "patrimonio_medio_medio_clusters": perfil_clusters["patrimonio_medio"].mean(),
                "patrimonio_medio_min_clusters": perfil_clusters["patrimonio_medio"].min(),
                "patrimonio_medio_max_clusters": perfil_clusters["patrimonio_medio"].max(),

                "qtd_bens_media_clusters": perfil_clusters["qtd_bens_media"].mean(),
                "qtd_bens_min_clusters": perfil_clusters["qtd_bens_media"].min(),
                "qtd_bens_max_clusters": perfil_clusters["qtd_bens_media"].max(),

                "qtd_macros_media_clusters": perfil_clusters["qtd_macros_media"].mean(),
                "qtd_macros_min_clusters": perfil_clusters["qtd_macros_media"].min(),
                "qtd_macros_max_clusters": perfil_clusters["qtd_macros_media"].max(),

                "concentracao_media_clusters": perfil_clusters["concentracao_media"].mean(),
                "concentracao_min_clusters": perfil_clusters["concentracao_media"].min(),
                "concentracao_max_clusters": perfil_clusters["concentracao_media"].max(),

                "macro_dominante_media_pct": perfil_clusters["macro_1_pct"].mean(),
                "macro_dominante_min_pct": perfil_clusters["macro_1_pct"].min(),
                "macro_dominante_max_pct": perfil_clusters["macro_1_pct"].max(),
            }

            lista_metricas.append(metricas_k)
            lista_perfis.append(perfil_clusters)

    metricas_busca_k = pd.DataFrame(lista_metricas)

    if len(lista_perfis) > 0:
        perfis_busca_k = pd.concat(lista_perfis, ignore_index=True)
    else:
        perfis_busca_k = pd.DataFrame()

    return metricas_busca_k, perfis_busca_k

In [64]:
# ============================================================
# EXECUÇÃO OPCIONAL DA BUSCA DE MÉTRICAS
# ============================================================

if RODAR_BUSCAS_METRICAS:

    metricas_busca_k, perfis_busca_k = avaliar_k_por_estrato(
        features_candidatos=features_candidatos,
        estratos_ordem=ESTRATOS_ORDEM,
        ranges_k=RANGES_K,
        colunas_percentuais=colunas_percentuais,
        random_state=RANDOM_STATE,
        n_init=N_INIT_KMEANS
    )

    display(metricas_busca_k)
    display(perfis_busca_k)

else:
    print("Busca exploratória de métricas desativada.")

Busca exploratória de métricas desativada.


In [65]:
if RODAR_BUSCAS_METRICAS:

    colunas_resumo_busca = [
        "estrato_patrimonial",
        "k",
        "tipo_matriz",
        "n_candidatos",
        "silhouette",
        "davies_bouldin",
        "calinski_harabasz",
        "menor_cluster",
        "perc_menor_cluster",
        "maior_cluster",
        "perc_maior_cluster",
        "qtd_bens_media_clusters",
        "qtd_macros_media_clusters",
        "concentracao_media_clusters",
        "macro_dominante_media_pct",
    ]

    metricas_busca_k_resumo = (
        metricas_busca_k[colunas_resumo_busca]
        .sort_values(["estrato_patrimonial", "k"])
        .reset_index(drop=True)
    )

    metricas_busca_k_resumo

In [66]:
if RODAR_BUSCAS_METRICAS:

    metricas_busca_k_resumo_formatada = metricas_busca_k_resumo.copy()

    for col in [
        "silhouette",
        "davies_bouldin",
        "calinski_harabasz",
        "qtd_bens_media_clusters",
        "qtd_macros_media_clusters",
        "concentracao_media_clusters",
        "macro_dominante_media_pct",
    ]:
        metricas_busca_k_resumo_formatada[col] = (
            metricas_busca_k_resumo_formatada[col].round(3)
        )

    for col in ["perc_menor_cluster", "perc_maior_cluster"]:
        metricas_busca_k_resumo_formatada[col] = (
            metricas_busca_k_resumo_formatada[col].map(lambda x: f"{x:.2%}")
        )

    metricas_busca_k_resumo_formatada

In [67]:
if RODAR_BUSCAS_METRICAS:

    pasta_saida = Path(PASTA_AUDITORIA_CLUSTERS)
    pasta_saida.mkdir(parents=True, exist_ok=True)

    metricas_busca_k.to_csv(
        pasta_saida / "metricas_busca_k.csv",
        sep=";",
        index=False,
        encoding="utf-8-sig"
    )

    perfis_busca_k.to_csv(
        pasta_saida / "perfis_busca_k.csv",
        sep=";",
        index=False,
        encoding="utf-8-sig"
    )

    print("Arquivos da busca exploratória salvos em:", pasta_saida.resolve())